# WoundWatch — Gemma 4 Fine-tuning with Unsloth

**Kaggle 실행 전 체크리스트:**
1. Settings → Accelerator: **GPU T4 x2** (또는 P100)
2. Add Data → `laithjj/diabetic-foot-ulcer-dfu`
3. Add Data → `leoscode/wound-segmentation-images`
4. Secrets → `HF_TOKEN` (HuggingFace Write 권한 토큰)
5. Settings → Internet → ON (패키지 설치 필요)

**목표:** Gemma 4 4B 멀티모달을 DFU 이미지 분석 태스크에 파인튜닝  
**출력 포맷:** ai_service.py와 동일한 JSON (infection, ischemia, severity, wound_area_cm2, description, confidence)  
**저장:** GGUF (Ollama) + HuggingFace Hub

In [ ]:
# ── 1. 의존성 설치 ────────────────────────────────────────────────────────────
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
    "trl>=0.9.0", "peft", "accelerate", "bitsandbytes"
], check=True)

print("설치 완료")

In [ ]:
# ── 2. 임포트 ────────────────────────────────────────────────────────────────
import json
import random
import re
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
from datasets import Dataset

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

random.seed(42)
print("임포트 완료")

In [ ]:
# ── 3. 시스템 프롬프트 (ai_service.py와 동일하게 유지) ────────────────────────
# 학습 시 이 프롬프트를 system turn에 넣어 모델이 JSON 출력을 학습하도록 함

SYSTEM_PROMPT = """You are a clinical AI assistant specialized in diabetic foot ulcer assessment.
Analyze the wound image and respond ONLY with a valid JSON object. No explanation, no markdown, no code fences.

{
  "infection": true or false,
  "ischemia": true or false,
  "severity": 0.0 to 10.0,
  "wound_area_cm2": estimated area as float,
  "description": "one paragraph clinical description in English",
  "confidence": 0.0 to 1.0
}

Assessment criteria:
- infection: Look for erythema, purulent discharge, warmth indicators, tissue necrosis, perilesional inflammation
- ischemia: Look for pallor, cyanosis, lack of granulation tissue, dry necrosis, pale wound bed
- severity: 0=minimal/healing, 5=moderate progression, 10=critical/limb-threatening
- wound_area_cm2: estimate based on proportion of visible foot area (average adult foot ~150 cm2)
- confidence: your confidence in the assessment (0=very uncertain, 1=highly confident)"""

USER_PROMPT = "Analyze this diabetic foot wound image and provide a structured clinical assessment."

print("시스템 프롬프트 설정 완료")

In [ ]:
# ── 4. 데이터셋 경로 확인 ────────────────────────────────────────────────────
DFU_DIR = Path("/kaggle/input/diabetic-foot-ulcer-dfu")
SEG_DIR = Path("/kaggle/input/wound-segmentation-images")

print("=== DFU Binary Dataset ===")
if DFU_DIR.exists():
    for p in sorted(DFU_DIR.iterdir())[:15]:
        count = len(list(p.glob("*.jpg"))) + len(list(p.glob("*.png"))) if p.is_dir() else 0
        print(f"  {p.name}/  ({count} images)")
else:
    print("  [없음] Kaggle에 데이터셋을 추가하세요")

print("\n=== Segmentation Dataset ===")
if SEG_DIR.exists():
    for p in sorted(SEG_DIR.rglob("*"))[:15]:
        print(f"  {p.relative_to(SEG_DIR)}")
else:
    print("  [없음] Kaggle에 데이터셋을 추가하세요")

In [ ]:
# ── 5. JSON 응답 생성 유틸 ────────────────────────────────────────────────────
# ai_service.py의 parse_gemma_response와 동일한 필드 구조

ULCER_KEYWORDS   = {"ulcer", "dfu", "wound", "positive", "diabetic"}
HEALTHY_KEYWORDS = {"healthy", "normal", "non_dfu", "negative", "control"}


def build_json_response(is_ulcer: bool, wound_area_ratio: float = 0.0) -> str:
    """학습용 JSON 응답 생성. ai_service.py parse_gemma_response와 동일한 필드."""
    if not is_ulcer:
        resp = {
            "infection": False,
            "ischemia": False,
            "severity": round(random.uniform(0.5, 1.5), 1),
            "wound_area_cm2": 0.0,
            "description": (
                "No wound detected. Foot skin appears intact with no signs of ulceration, "
                "necrosis, or inflammatory changes. Peripheral tissue looks healthy. "
                "Continue routine monitoring."
            ),
            "confidence": round(random.uniform(0.85, 0.95), 2),
        }
        return json.dumps(resp, indent=2)

    # 면적 비율 → cm² (성인 발 평균 150 cm²)
    area_cm2 = round(max(wound_area_ratio * 150, 0.5), 1)

    if wound_area_ratio < 0.02:       # 소형 (~0–3 cm²)
        infection = random.random() < 0.35
        ischemia  = False
        severity  = round(random.uniform(2.0, 4.0), 1)
        desc = (
            f"Small diabetic foot ulcer observed with limited wound extent ({area_cm2} cm²). "
            "Early-stage lesion with minimal surrounding tissue involvement."
        )
    elif wound_area_ratio < 0.05:     # 중형 (~3–7.5 cm²)
        infection = random.random() < 0.65
        ischemia  = random.random() < 0.30
        severity  = round(random.uniform(4.0, 6.5), 1)
        desc = (
            f"Moderate diabetic foot ulcer ({area_cm2} cm²) with progressive tissue involvement. "
            "Wound margins show moderate erythema."
        )
    else:                              # 대형 (>7.5 cm²)
        infection = True
        ischemia  = random.random() < 0.60
        severity  = round(random.uniform(6.5, 9.5), 1)
        desc = (
            f"Significant diabetic foot ulcer with extensive tissue damage ({area_cm2} cm²). "
            "High risk of systemic complications."
        )

    if infection:
        desc += " Erythema and purulent exudate present, consistent with bacterial colonization."
    if ischemia:
        desc += " Compromised peripheral blood flow noted with pale wound bed and dry necrotic margins."
    if severity >= 7.0:
        desc += " Immediate medical attention strongly recommended to prevent limb loss."
    elif severity >= 5.0:
        desc += " Urgent clinical evaluation advised within 48 hours."
    else:
        desc += " Close outpatient monitoring recommended."

    resp = {
        "infection": infection,
        "ischemia": ischemia,
        "severity": severity,
        "wound_area_cm2": area_cm2,
        "description": desc,
        "confidence": round(random.uniform(0.68, 0.85), 2),
    }
    return json.dumps(resp, indent=2)


def make_conversation(image_path: str, is_ulcer: bool, wound_area_ratio: float = 0.0) -> dict:
    """Gemma 4 멀티모달 chat format.

    모든 role의 content를 list[dict]로 통일.
    Dataset.from_list은 동일 컬럼 내 타입이 일치해야 하므로
    str/list 혼용 시 ArrowInvalid: cannot mix list and non-list 에러 발생.
    """
    abs_path = str(Path(image_path).resolve())
    return {
        "messages": [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}],
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": f"file://{abs_path}"},
                    {"type": "text",  "text": USER_PROMPT},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": build_json_response(is_ulcer, wound_area_ratio)}],
            },
        ]
    }


print("유틸 함수 로드 완료")
# 샘플 확인
print("\n[정상 발 샘플]")
print(json.loads(build_json_response(False))["description"][:80])
print("\n[궤양 샘플 (대형)]")
print(build_json_response(True, 0.08)[:200])

In [ ]:
# ── 6. 데이터 로드 ────────────────────────────────────────────────────────────
samples = []

# ── Dataset 1: laithjj/diabetic-foot-ulcer-dfu (폴더 기반 이진 분류) ──
if DFU_DIR.exists():
    dfu_count = 0
    for folder in DFU_DIR.rglob("*"):
        if not folder.is_dir():
            continue
        name = folder.name.lower()
        if any(k in name for k in ULCER_KEYWORDS):
            label = True
        elif any(k in name for k in HEALTHY_KEYWORDS):
            label = False
        else:
            continue
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            for img in folder.glob(ext):
                samples.append(make_conversation(str(img), is_ulcer=label))
                dfu_count += 1
    print(f"DFU binary: {dfu_count}개")
else:
    print("[SKIP] DFU binary 데이터셋 없음")

# ── Dataset 2: leoscode/wound-segmentation-images (마스크 기반 면적) ──
if SEG_DIR.exists():
    image_dir = next((d for d in SEG_DIR.rglob("images") if d.is_dir()), None)
    mask_dir  = next((d for d in SEG_DIR.rglob("masks")  if d.is_dir()), None)

    if image_dir and mask_dir:
        seg_count = 0
        for img_path in sorted(image_dir.glob("*.jpg")):
            mask_path = mask_dir / f"{img_path.stem}.png"
            if not mask_path.exists():
                mask_path = mask_dir / f"{img_path.stem}.jpg"
            if mask_path.exists():
                mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
                ratio = np.count_nonzero(binary) / binary.size
            else:
                ratio = 0.05  # 마스크 없으면 중간값
            samples.append(make_conversation(str(img_path), is_ulcer=True, wound_area_ratio=ratio))
            seg_count += 1
        print(f"Segmentation: {seg_count}개")
    else:
        print("[SKIP] Segmentation images/masks 폴더 구조 확인 필요")
else:
    print("[SKIP] Segmentation 데이터셋 없음")

print(f"\n총 샘플: {len(samples)}개")

if len(samples) == 0:
    print("\n[경고] 샘플이 없습니다. 데이터셋을 Kaggle에 추가했는지 확인하세요.")
    raise SystemExit(0)

In [ ]:
# ── 7. Train / Val 분할 → HuggingFace Dataset ────────────────────────────────
random.shuffle(samples)
split     = int(len(samples) * 0.85)
train_raw = samples[:split]
val_raw   = samples[split:]

train_dataset = Dataset.from_list(train_raw)
val_dataset   = Dataset.from_list(val_raw)

print(f"Train: {len(train_dataset)}개 / Val: {len(val_dataset)}개")
print("\n샘플 확인 (messages 구조):")
sample = train_dataset[0]
for msg in sample["messages"]:
    role    = msg["role"]
    content = msg["content"]   # list[dict], Arrow가 None으로 채운 키 존재 가능
    types   = [c["type"] for c in content]
    # Arrow 직렬화 후 없는 키는 None으로 채워지므로 or로 안전하게 읽음
    first   = content[0]
    preview = (first.get("text") or first.get("image") or "")[:60]
    print(f"  [{role}] types={types}  preview={preview!r}")

In [ ]:
# ── 8. 모델 로드 + LoRA 설정 ─────────────────────────────────────────────────
#
# Gemma 4 실제 모델 ID (unsloth 컬렉션 확인):
#   unsloth/gemma-4-E2B-it   ← 5B, T4 x2 권장  ★
#   unsloth/gemma-4-E4B-it   ← 8B, P100 / A100 이상
#
# BNB 4bit 사전 양자화 버전(-bnb-4bit)은 미출시.
# load_in_4bit=True 로 런타임 양자화.

MODEL_ID    = "unsloth/gemma-4-E2B-it"   # T4 기본값 (5B)
MAX_SEQ_LEN = 2048

model, processor = FastVisionModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
# ── 9. Trainer 설정 ───────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,       # effective batch = 8
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=True,
        max_seq_length=MAX_SEQ_LEN,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        output_dir="/kaggle/working/woundwatch-checkpoints",
        report_to="none",
        remove_unused_columns=False,
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)

print("Trainer 준비 완료")

In [ ]:
# ── 10. 파인튜닝 실행 ─────────────────────────────────────────────────────────
trainer_stats = trainer.train()

print(f"\n훈련 완료")
print(f"총 스텝: {trainer_stats.global_step}")
print(f"최종 Loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# ── 11. 추론 테스트 — JSON 출력 검증 ─────────────────────────────────────────
import json, re

FastVisionModel.for_inference(model)

# 첫 번째 val 샘플로 테스트
test_sample = val_raw[0]
img_uri  = test_sample["messages"][1]["content"][0]["image"]   # file:// URI
img_path = img_uri.replace("file://", "")
test_image = Image.open(img_path).convert("RGB")

# content를 모두 list[dict]로 통일 — apply_chat_template이 content를 순회하기 때문
messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_PROMPT}],
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": test_image},
            {"type": "text",  "text": USER_PROMPT},
        ],
    },
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
)
raw_text = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("=== Raw output ===")
print(raw_text)

# JSON 파싱 검증 (ai_service.parse_gemma_response와 동일 로직)
match = re.search(r'\{.*\}', raw_text, re.DOTALL)
if match:
    try:
        parsed = json.loads(match.group())
        required = ["infection", "ischemia", "severity", "wound_area_cm2", "description", "confidence"]
        missing  = [k for k in required if k not in parsed]
        print("\n=== 파싱 성공 ===")
        print(f"infection:  {parsed.get('infection')}")
        print(f"ischemia:   {parsed.get('ischemia')}")
        print(f"severity:   {parsed.get('severity')}")
        print(f"area cm²:   {parsed.get('wound_area_cm2')}")
        print(f"confidence: {parsed.get('confidence')}")
        if missing:
            print(f"[경고] 누락 필드: {missing}")
        else:
            print("[OK] 모든 필드 존재")
    except json.JSONDecodeError as e:
        print(f"[실패] JSON 파싱 오류: {e}")
else:
    print("[실패] JSON 블록을 찾을 수 없음 — 추가 학습 필요")

In [ ]:
# ── 12. 모델 저장 ─────────────────────────────────────────────────────────────
import subprocess, shutil, os
from pathlib import Path

SAVE_DIR     = "/tmp/woundwatch-gemma4"
GGUF_TMP_DIR = "/tmp/woundwatch-gemma4-gguf"

# 체크포인트 삭제
ckpt_dir = "/kaggle/working/woundwatch-checkpoints"
if os.path.exists(ckpt_dir):
    shutil.rmtree(ckpt_dir)
    print(f"체크포인트 삭제: {ckpt_dir}")

# (A) LoRA 어댑터 저장
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f"LoRA 어댑터 저장 완료: {SAVE_DIR}")

# (B) GGUF 변환
# llama.cpp 변환기가 --outfile을 상대경로로 씀 → CWD가 /kaggle/working(꽉 참)이면 실패
# CWD를 /tmp로 바꿔서 중간 F16 파일도 /tmp에 쓰이도록 함
print("\nGGUF 변환 시작 (CWD=/tmp)...")
os.chdir("/tmp")
try:
    model.save_pretrained_gguf(
        GGUF_TMP_DIR,
        processor,
        quantization_method="q4_k_m",
    )
    gguf_files = list(Path(GGUF_TMP_DIR).glob("*.gguf"))
    for f in gguf_files:
        print(f"  GGUF 완료: {f.name} ({f.stat().st_size / 1024**2:.0f} MB)")
except RuntimeError as e:
    gguf_files = []
    print(f"[GGUF 건너뜀] {e}")
finally:
    os.chdir("/kaggle/working")  # CWD 복구

In [ ]:
# ── 13. Modelfile 확인 및 SYSTEM_PROMPT 반영 ─────────────────────────────────
# Unsloth는 출력 디렉토리에 _gguf suffix를 붙임
GGUF_OUT_DIR = Path(GGUF_TMP_DIR + "_gguf")

gguf_model   = GGUF_OUT_DIR / "gemma-4-e2b-it.Q4_K_M.gguf"
gguf_mmproj  = GGUF_OUT_DIR / "gemma-4-e2b-it.F16-mmproj.gguf"
modelfile_path = GGUF_OUT_DIR / "Modelfile"

print("생성된 파일:")
for f in GGUF_OUT_DIR.iterdir():
    print(f"  {f.name}  ({f.stat().st_size / 1024**2:.0f} MB)")

# Unsloth가 자동 생성한 Modelfile에 SYSTEM_PROMPT 덮어쓰기
modelfile_content = f"""FROM {gguf_model.name}
SYSTEM \"\"\"
{SYSTEM_PROMPT}
\"\"\"

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER num_predict 256
"""

modelfile_path.write_text(modelfile_content)
print(f"\nModelfile 업데이트: {modelfile_path}")
print("\n로컬 Ollama 등록 방법:")
print(f"  # GGUF 파일 다운로드 후:")
print(f"  ollama create woundwatch -f Modelfile")
print(f"  ollama run woundwatch")
print(f"\n  # 또는 HF 업로드 후:")
print(f"  ollama run hf.co/5seoyoung/woundwatch-gemma4-dfu:Q4_K_M")

In [ ]:
# ── 14. HuggingFace Hub 업로드 ────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

HF_USERNAME = "5seoyoung"
HF_REPO     = f"{HF_USERNAME}/woundwatch-gemma4-dfu"
api         = HfApi()

# (A) LoRA 어댑터 업로드 (이미 완료됐으면 스킵됨)
model.push_to_hub(HF_REPO, token=hf_token)
processor.push_to_hub(HF_REPO, token=hf_token)
print(f"LoRA 업로드 완료: https://huggingface.co/{HF_REPO}")

# (B) 이미 변환된 GGUF 파일 직접 업로드 (재변환 없음)
# push_to_hub_gguf는 내부에서 재변환 → 또 실패하므로 직접 업로드
gguf_model  = GGUF_OUT_DIR / "gemma-4-e2b-it.Q4_K_M.gguf"
gguf_mmproj = GGUF_OUT_DIR / "gemma-4-e2b-it.F16-mmproj.gguf"

for local_path, repo_name in [
    (gguf_model,  "gemma-4-e2b-it.Q4_K_M.gguf"),
    (gguf_mmproj, "gemma-4-e2b-it.F16-mmproj.gguf"),
    (GGUF_OUT_DIR / "Modelfile", "Modelfile"),
]:
    if not local_path.exists():
        print(f"[건너뜀] {local_path.name} 없음")
        continue
    size_mb = local_path.stat().st_size / 1024**2
    print(f"업로드 중: {repo_name} ({size_mb:.0f} MB)...")
    api.upload_file(
        path_or_fileobj=str(local_path),
        path_in_repo=repo_name,
        repo_id=HF_REPO,
        token=hf_token,
    )
    print(f"  완료: {repo_name}")

print(f"\n✅ 모든 업로드 완료!")
print(f"HF 모델: https://huggingface.co/{HF_REPO}")
print(f"Ollama:  ollama run hf.co/{HF_REPO}:Q4_K_M")

## 업로드 후 연결 방법

### 방법 A — HuggingFace 직접 로드 (백엔드)
```python
# backend/app/services/ai_service.py
from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    "5seoyoung/woundwatch-gemma4-dfu",
    load_in_4bit=True,
)
FastVisionModel.for_inference(model)
```

### 방법 B — Ollama 로컬 실행
```bash
# HF에서 바로 실행 (GGUF 자동 다운로드)
ollama run hf.co/5seoyoung/woundwatch-gemma4-dfu:Q4_K_M

# 또는 직접 등록
ollama create woundwatch -f Modelfile
ollama run woundwatch
```

### .env.production 업데이트
```
VITE_API_URL=https://5seoyoung-woundwatch.hf.space
```